In [8]:
# ...existing code...
# Quick test cell: embed two short proteins and write a tiny HDF5 to validate outputs.
# Uses Facebook / HuggingFace "esm" package (fair-esm) esm1b model instead of esme.

import torch
import numpy as np
import h5py
import esm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load model + alphabet from fair-esm / huggingface esm repo
model, alphabet = esm.pretrained.esm1b_t33_650M_UR50S()
model = model.to(device)
model.eval()
batch_converter = alphabet.get_batch_converter()

sequences = {
    "P_TEST1": "MEEPQSDPSVEPPLSQESTFSLDLWK",
    "P_TEST2": "MADQLTEEQIAEFKEAFSLFDKDG"
}

# prepare batch in the (id, seq) format expected by batch_converter
batch = list(sequences.items())  # list of (id, seq)
labels, strs, tokens = batch_converter(batch)
tokens = tokens.to(device)

with torch.no_grad():
    # request final-layer representations (layer index 33 for esm1b_t33_650M_UR50S)
    results = model(tokens, repr_layers=[33], return_contacts=True)
    reps = results["representations"][33]  # shape (batch, seq_len, embed_dim)

h5path = "swissprot_esm1b_test.h5"
with h5py.File(h5path, "w") as h5f:
    for i, (seq_id, seq) in enumerate(batch):
        L = len(seq)
        # remove BOS token at position 0 and EOS at -1
        res_emb = reps[i, 1 : 1 + L, :].cpu().numpy().astype("float32")
        # contact map proxy: cosine similarity
        v = res_emb
        norm = np.linalg.norm(v, axis=1, keepdims=True)
        norm[norm == 0] = 1.0
        vnorm = v / norm
        contact = vnorm.dot(vnorm.T).astype("float32")

        grp = h5f.create_group(seq_id)
        grp.create_dataset("embeddings", data=res_emb, compression="gzip", compression_opts=4)
        grp.create_dataset("contacts", data=contact, compression="gzip", compression_opts=4)
        grp.attrs["sequence"] = seq

print("Test HDF5 written:", h5path)


Test HDF5 written: swissprot_esm1b_test.h5


In [43]:
results

{'logits': tensor([[[ 20.2196,  -8.7814,   2.3032,  ..., -13.8319, -15.4220,  -8.5840],
          [-13.2152, -16.9361,  -7.7244,  ..., -16.0511, -16.0504, -16.9744],
          [-15.0968, -17.8301, -11.7673,  ..., -16.3562, -16.5059, -17.8216],
          ...,
          [-14.5625, -17.7961, -10.2264,  ..., -16.4952, -16.4259, -17.7174],
          [-14.2917, -17.5896, -10.8455,  ..., -16.4694, -16.5263, -17.5750],
          [ -6.7716, -14.2414,  30.8424,  ..., -15.1069, -16.0993, -14.5719]],
 
         [[ 19.2694,  -7.5755,   4.7198,  ..., -13.9826, -15.2548,  -7.3058],
          [-14.7090, -18.7142, -11.2430,  ..., -16.4069, -16.4881, -18.8483],
          [-17.2637, -19.0268, -15.8023,  ..., -16.7391, -16.6093, -18.9785],
          ...,
          [ -5.6132, -12.2658,  29.7338,  ..., -14.7550, -15.4611, -12.6185],
          [ -0.2394,  -4.3899,  60.5192,  ..., -14.0755, -15.1042,  -5.2811],
          [ -0.2394,  -4.3899,  60.5192,  ..., -14.0755, -15.1042,  -5.2811]]],
        device='cud

In [47]:
batch_embeddings[0].shape

torch.Size([26, 26])

In [44]:

h5f = h5py.File("swissprot_esm1b_test.h5", "r")
selected_prot_ids = ["P_TEST1", "P_TEST2"]  # Example batch

batch_embeddings = []
for prot_id in selected_prot_ids:
    emb = torch.tensor(h5f[prot_id]["contacts"][:])  # Convert numpy back to torch tensor
    batch_embeddings.append(emb)

print(batch_embeddings)

[tensor([[1.0000, 0.9235, 0.9217, 0.9263, 0.9244, 0.9194, 0.9202, 0.9182, 0.9203,
         0.9239, 0.9189, 0.9249, 0.9233, 0.9173, 0.9160, 0.9237, 0.9144, 0.9146,
         0.9094, 0.9141, 0.9119, 0.9158, 0.9051, 0.9143, 0.9190, 0.9184],
        [0.9235, 1.0000, 0.9789, 0.9492, 0.9489, 0.9445, 0.9431, 0.9401, 0.9428,
         0.9454, 0.9735, 0.9444, 0.9424, 0.9373, 0.9404, 0.9492, 0.9718, 0.9405,
         0.9319, 0.9380, 0.9361, 0.9355, 0.9331, 0.9319, 0.9481, 0.9426],
        [0.9217, 0.9789, 1.0000, 0.9578, 0.9574, 0.9515, 0.9567, 0.9510, 0.9491,
         0.9497, 0.9876, 0.9600, 0.9516, 0.9484, 0.9470, 0.9502, 0.9806, 0.9475,
         0.9385, 0.9431, 0.9427, 0.9391, 0.9362, 0.9408, 0.9507, 0.9435],
        [0.9263, 0.9492, 0.9578, 1.0000, 0.9577, 0.9542, 0.9532, 0.9855, 0.9509,
         0.9505, 0.9505, 0.9875, 0.9865, 0.9471, 0.9499, 0.9506, 0.9473, 0.9490,
         0.9417, 0.9461, 0.9425, 0.9402, 0.9355, 0.9401, 0.9510, 0.9436],
        [0.9244, 0.9489, 0.9574, 0.9577, 1.0000, 0.9503

In [5]:
# ...existing code...
# Full pipeline (fair-esm esm1b) with sliding-window support for sequences longer than model max.
# Requires: pip install fair-esm h5py biopython tqdm

import torch
import numpy as np
import h5py
from Bio import SeqIO
from tqdm.auto import tqdm

import esm  # fair-esm / huggingface esm package

# Config
fasta_path = "/home/atoffano/PFP_baselines/data/swissprot/2024_01/swissprot_2024_01.fasta"
out_h5 = "swissprot_esm1b.h5"
batch_size = 8               # number of sequences (or windows) per model batch
max_sequence_length = None   # skip sequences longer than this (optional)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Count sequences and total residues for size/ETA estimation
print("Counting sequences in fasta...")
num_seqs = 0
total_residues = 0
with open(fasta_path, "r") as fh:
    for rec in SeqIO.parse(fh, "fasta"):
        num_seqs += 1
        total_residues += len(rec.seq)
print(f"Found {num_seqs} sequences, {total_residues} total residues")

# Load model + alphabet from fair-esm (HuggingFace)
print("Loading esm1b model (fair-esm)...")
model, alphabet = esm.pretrained.esm1b_t33_650M_UR50S()
model = model.to(device)
model.eval()
batch_converter = alphabet.get_batch_converter()

# infer model token limit (reserve BOS/EOS)
max_model_tokens = None
if hasattr(model, "args") and getattr(model.args, "max_positions", None) is not None:
    max_model_tokens = int(model.args.max_positions)
else:
    # sensible default if unknown
    max_model_tokens = 1024
window_seq_len = max_model_tokens - 2  # reserve for BOS/EOS inside model tokens
# define stride (50% overlap)
stride = max(1, window_seq_len // 2)

def embed_sequence_with_sliding(seq_id, seq, h5f, processed_stats):
    """
    Embed a sequence possibly longer than the model window using sliding windows.
    Aggregates overlapping window embeddings by averaging per-residue embeddings.
    Stores embeddings and contact map in h5f under group seq_id.
    Updates processed_stats (dict with keys 'bytes','residues','seqs').
    """
    L = len(seq)
    # quick path: fits in one window
    if L <= window_seq_len:
        batch = [(seq_id, seq)]
        labels, strs, tokens = batch_converter(batch)
        tokens = tokens.to(device)
        with torch.no_grad():
            # request contacts from model when available
            results = model(tokens, repr_layers=[33], return_contacts=False)
            reps = results["representations"][33]  # (1, seq_len_with_specials, D)
        emb = reps[0, 1 : 1 + L, :].cpu().numpy().astype("float16")  # (L, D)

    else:
        # sliding windows
        windows = []
        starts = list(range(0, L, stride))
        # ensure last window ends at L
        if starts[-1] + window_seq_len < L:
            starts.append(L - window_seq_len)
        starts = sorted(set([s for s in starts if s >= 0]))
        for s in starts:
            e = min(s + window_seq_len, L)
            windows.append((s, e, seq[s:e]))
        # accumulate embeddings and counts
        acc = None
        counts = np.zeros((L,), dtype=np.int32)
        # process windows in batches
        for i in range(0, len(windows), batch_size):
            batch_windows = windows[i : i + batch_size]
            batch = [(f"{seq_id}_w{i+j}", w[2]) for j, w in enumerate(batch_windows)]
            labels, strs, tokens = batch_converter(batch)
            tokens = tokens.to(device)
            with torch.no_grad():
                results = model(tokens, repr_layers=[33], return_contacts=False)
                reps = results["representations"][33].cpu().numpy()  # (batch_w, seq_len_with_specials, D)
            for j, (s, e, _) in enumerate(batch_windows):
                sub_len = e - s
                sub_emb = reps[j, 1 : 1 + sub_len, :]  # (sub_len, D)
                if acc is None:
                    D = sub_emb.shape[-1]
                    acc = np.zeros((L, D), dtype=np.float64)
                acc[s:e, :] += sub_emb
                counts[s:e] += 1
        # finalize averaged embeddings
        # sanity: avoid divide by zero (shouldn't happen)
        counts = counts.astype(np.float32)
        counts[counts == 0] = 1.0
        emb = (acc / counts[:, None]).astype("float16")  # (L, D)


    # store in hdf5
    grp = h5f.create_group(seq_id)
    grp.create_dataset("embeddings", data=emb, compression="gzip", compression_opts=4)
    grp.attrs["sequence"] = seq
    grp.attrs["length"] = L

    # update stats
    processed_stats["bytes"] += emb.nbytes
    processed_stats["residues"] += L
    processed_stats["seqs"] += 1

# Stream fasta, embed in batches, write HDF5 and update tqdm with ETA and estimated final size (Go)
processed = {"bytes": 0, "residues": 0, "seqs": 0}
with h5py.File(out_h5, "w") as h5f:
    pbar = tqdm(total=num_seqs, desc="Embedding sequences", unit="seq")
    batch_records = []
    with open(fasta_path, "r") as fh:
        for rec in SeqIO.parse(fh, "fasta"):
            L = len(rec.seq)
            if max_sequence_length and L > max_sequence_length:
                pbar.update(1)
                continue
            # we process per-sequence (embedding may internally do windows and batching); to keep memory stable,
            # handle each sequence individually here (you can adjust to group short sequences into model batches).
            try:
                embed_sequence_with_sliding(rec.id, str(rec.seq), h5f, processed)
            except Exception as e:
                print(f"Error embedding {rec.id}: {e}")
            # update progress and estimate final HDF5 size
            pbar.update(1)
            est_gb = None
            if processed["residues"] > 0:
                per_res_bytes = processed["bytes"] / processed["residues"]
                est_total_bytes = per_res_bytes * total_residues
                est_gb = est_total_bytes / 1e9  # Go = 1e9 bytes
            pbar.set_postfix({"est_h5": f"{est_gb:.2f} Go" if est_gb is not None else "calculating"})
    pbar.close()

print("Done. HDF5 written:", out_h5)
# ...existing code...

Counting sequences in fasta...
Found 570830 sequences, 206533160 total residues
Loading esm1b model (fair-esm)...


Embedding sequences:   0%|          | 592/570830 [01:19<73:33:08,  2.15seq/s, est_h5=528.72 Go] 

KeyboardInterrupt: 

In [7]:
# Quick test: synthetic 10000-aa protein -> embed using embed_sequence_with_sliding
motif = "MEEPQSDPSVEPPLSQESTFSLDLWK"
target_length = 10000
repeats = (target_length + len(motif) - 1) // len(motif)
long_sequence = (motif * repeats)[:target_length]

out_h5 = "long_protein_10000_esm1b_test.h5"
processed = {"bytes": 0, "residues": 0, "seqs": 0}
with h5py.File(out_h5, "w") as h5f:
    embed_sequence_with_sliding("LONG_10000", long_sequence, h5f, processed)

print("Wrote embeddings for LONG_10000 to", out_h5)


Wrote embeddings for LONG_10000 to long_protein_10000_esm1b_test.h5


In [ ]:
import torch
import numpy as np
import h5py
from Bio import SeqIO
from tqdm.auto import tqdm

import esm  # fair-esm / huggingface esm package


# Config
fasta_path = "/lustre/fsn1/projects/rech/dqy/uki62ne/tempdata/PFP_layer/data/swissprot/2024_01/swissprot_2024_01.fasta"
out_h5 = "swissprot_esm1b_per_aa.h5"
batch_size = 8               # number of sequences (or windows) per model batch
max_sequence_length = None   # skip sequences longer than this (optional)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
checkpoint = (
    "/lustre/fswork/projects/rech/dqy/uki62ne/torch_cache/esm1b_t33_650M_UR50S.pt"
)

# Count sequences and total residues for size/ETA estimation
print("Counting sequences in fasta...")
num_seqs = 0
total_residues = 0
with open(fasta_path, "r") as fh:
    for rec in SeqIO.parse(fh, "fasta"):
        num_seqs += 1
        total_residues += len(rec.seq)
print(f"Found {num_seqs} sequences, {total_residues} total residues")

# Load model + alphabet from local checkpoint
print("Loading model...")
model, alphabet = esm.pretrained.load_model_and_alphabet_local(checkpoint)
model = model.to(device)
model.eval()
batch_converter = alphabet.get_batch_converter()

# infer model token limit (reserve BOS/EOS)
max_model_tokens = None
if hasattr(model, "args") and getattr(model.args, "max_positions", None) is not None:
    max_model_tokens = int(model.args.max_positions)
window_seq_len = max_model_tokens - 2  # reserve for BOS/EOS inside model tokens
# define stride (50% overlap)
stride = max(1, window_seq_len // 2)

def embed_sequence_with_sliding(seq_id, seq, h5f, processed_stats):
    """
    Embed a sequence using the model window with optional sliding windows.
    Aggregates overlapping window embeddings by averaging per-residue embeddings.
    Stores embeddings and minimal metadata in h5f under group seq_id.
    Updates processed_stats (dict with keys 'bytes','residues','seqs').
    """
    L = len(seq) 

    # Fast path: sequence fits in a single model window
    if L <= window_seq_len:
        # prepare a single-item batch for the model in (id, seq) format
        batch = [(seq_id, seq)]
        labels, strs, tokens = batch_converter(batch)  # convert to token tensor with special tokens
        tokens = tokens.to(device)

        with torch.no_grad():
            results = model(tokens, repr_layers=[33], return_contacts=False)
            # results["representations"][33] shape: (batch=1, seq_len_with_specials, D)
            reps = results["representations"][33]

        # skip BOS (index 0) and take L residues; result is shape (L, D)
        emb = reps[0, 1 : 1 + L, :].cpu().numpy().astype("float16")

    else:
        # Sliding windows path: sequence longer than model window
        windows = []
        # generate start indices spaced by stride; covers from 0 .. L-1
        starts = list(range(0, L, stride))

        # ensure the last window ends exactly at L (so we don't leave tail residues uncovered)
        if starts[-1] + window_seq_len < L:
            starts.append(L - window_seq_len)

        # keep only non-negative starts and unique sorted values
        starts = sorted(set([s for s in starts if s >= 0]))

        # build (start, end, subsequence) tuples for each window
        for s in starts:
            e = min(s + window_seq_len, L)  # end is exclusive
            windows.append((s, e, seq[s:e]))

        # acc will accumulate summed embeddings for every residue position across windows
        acc = None
        # counts will track how many windows contributed to each residue position
        counts = np.zeros((L,), dtype=np.int32)

        # process windows in model batches to utilize batching and limit memory usage
        for i in range(0, len(windows), batch_size):
            batch_windows = windows[i : i + batch_size]  # slice out the next batch of windows

            # create batch entries like (seq_id_windowIndex, subsequence)
            batch = [(f"{seq_id}_w{i+j}", w[2]) for j, w in enumerate(batch_windows)]

            # convert batch to model tokens and move to device
            labels, strs, tokens = batch_converter(batch)
            tokens = tokens.to(device)

            # run model to get representations for each window in this batch
            with torch.no_grad():
                results = model(tokens, repr_layers=[33], return_contacts=False)
                # reps shape: (batch_w, seq_len_with_specials, D)
                reps = results["representations"][33].cpu().numpy()

            # iterate over windows in this model batch and add their embeddings to the accumulator
            for j, (s, e, _) in enumerate(batch_windows):
                sub_len = e - s  # number of residues in this window
                # extract per-residue embeddings for this window (skip BOS at 0)
                sub_emb = reps[j, 1 : 1 + sub_len, :]  # shape (sub_len, D)

                # initialize acc on first use with float64 for numerical stability while summing
                if acc is None:
                    D = sub_emb.shape[-1]
                    acc = np.zeros((L, D), dtype=np.float64)

                # add window embeddings into the corresponding positions [s:e] of acc
                acc[s:e, :] += sub_emb

                # increment counts for residues covered by this window
                counts[s:e] += 1

        # finalize averaged embeddings: avoid division by zero (shouldn't happen because windows cover all residues)
        counts = counts.astype(np.float32)
        counts[counts == 0] = 1.0  # guard
        # divide summed embeddings by counts per residue and convert to float16 for storage
        emb = (acc / counts[:, None]).astype("float16")  # shape (L, D)


    # store in hdf5
    grp = h5f.create_group(seq_id)
    grp.create_dataset("embeddings", data=emb, compression="gzip", compression_opts=4)
    grp.attrs["sequence"] = seq
    grp.attrs["length"] = L

    # update stats
    processed_stats["bytes"] += emb.nbytes
    processed_stats["residues"] += L
    processed_stats["seqs"] += 1

# Stream fasta, embed in batches, write HDF5 and update tqdm with ETA and estimated final size (Go)
processed = {"bytes": 0, "residues": 0, "seqs": 0}
with h5py.File(out_h5, "w") as h5f:
    pbar = tqdm(total=num_seqs, desc="Embedding sequences", unit="seq")
    batch_records = []
    with open(fasta_path, "r") as fh:
        for rec in SeqIO.parse(fh, "fasta"):
            L = len(rec.seq)
            if max_sequence_length and L > max_sequence_length:
                pbar.update(1)
                continue
            # we process per-sequence (embedding may internally do windows and batching); to keep memory stable,
            # handle each sequence individually here (you can adjust to group short sequences into model batches).
            try:
                embed_sequence_with_sliding(rec.id, str(rec.seq), h5f, processed)
            except Exception as e:
                print(f"Error embedding {rec.id}: {e}")
            # update progress and estimate final HDF5 size
            pbar.update(1)
            est_gb = None
            if processed["residues"] > 0:
                per_res_bytes = processed["bytes"] / processed["residues"]
                est_total_bytes = per_res_bytes * total_residues
                est_gb = est_total_bytes / 1e9  # Go = 1e9 bytes
            pbar.set_postfix({"est_h5": f"{est_gb:.2f} Go" if est_gb is not None else "calculating"})
    pbar.close()

print("Done. HDF5 written:", out_h5)